# Cardiac Patient Monitoring System
## 02 — Data Cleaning & Data Quality (Phase 2)

Input: `data/processed/heart_disease_cleveland_stage1.csv` (produced in notebook 01).

This notebook covers:
1. Missing values — quantify and decide a strategy (documented, not yet applied to modeling data)
2. Duplicates — inspect and decide
3. Invalid / out-of-range values per feature
4. Data type correction
5. Categorical variable identification + encoding plan
6. Outlier screening (IQR-based, no automatic removal)
7. Final data-quality report saved to `outputs/results/`

**Rule:** `data/raw/` is never touched. Any transformation here is either descriptive
(inspection) or produces a new file in `data/processed/`. Actual imputation/encoding/scaling
that must be *fit only on training data* is deferred to the Scikit-learn Pipeline in Phase 6 —
here we only decide and document the strategy.


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)

df = pd.read_csv("../data/processed/heart_disease_cleveland_stage1.csv")
df.shape


(303, 15)

## 1. Missing values — quantify


In [2]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_report[missing_report['missing_count'] > 0]


,missing_count,missing_pct
ca,4,1.32
thal,2,0.66


**Findings:**
- `ca` (number of major vessels colored by fluoroscopy): 4 missing (1.32%)
- `thal` (thallium test result): 2 missing (0.66%)
- No other column has missing values, including the target.

**Why they might exist:** both are results of specific diagnostic procedures (fluoroscopy,
thallium stress test) that may not have been performed or recorded for every patient — plausible
real-world missingness, not a data-entry artifact.

**Decision:** Given the very small proportion (<2% per column), we will use **median imputation
for `ca`** (integer count, right-skewed, few discrete values → median is robust) and **most-frequent
(mode) imputation for `thal`** (categorical). Both will be implemented as `SimpleImputer` steps
inside the Scikit-learn `ColumnTransformer` in Phase 6, fit exclusively on the training split to
avoid leakage. We do **not** impute here — this cell only documents the plan and previews the
effect for sanity-checking.


In [3]:
# Sanity preview only (not written back to any processed file)
preview_ca_median = df['ca'].median()
preview_thal_mode = df['thal'].mode().iloc[0]
print("Preview ca median:", preview_ca_median)
print("Preview thal mode:", preview_thal_mode)


Preview ca median: 0.0
Preview thal mode: 3.0


## 2. Duplicates


In [4]:
n_dupes = df.duplicated().sum()
print(f"Fully duplicated rows: {n_dupes}")


Fully duplicated rows: 0


**Decision:** No duplicate rows were found, so no removal action is required. If duplicates
had existed, we would have checked whether they represented genuinely repeated patient records
(likely erroneous, safe to drop) versus coincidentally identical but distinct patients (should be
kept) before deciding.


## 3. Invalid / out-of-range value checks

Each categorical/coded feature has a documented, finite value set. We check that observed values
fall within the documented ranges.


In [5]:
expected_values = {
    'sex': {0, 1},
    'cp': {1, 2, 3, 4},
    'fbs': {0, 1},
    'restecg': {0, 1, 2},
    'exang': {0, 1},
    'slope': {1, 2, 3},
    'ca': {0.0, 1.0, 2.0, 3.0},       # NaN allowed (missing)
    'thal': {3.0, 6.0, 7.0},          # NaN allowed (missing)
    'target': {0, 1},
    'num': {0, 1, 2, 3, 4},
}

invalid_report = {}
for col, allowed in expected_values.items():
    observed = set(df[col].dropna().unique())
    unexpected = observed - allowed
    invalid_report[col] = sorted(unexpected) if unexpected else "OK"

pd.Series(invalid_report)


sex        OK
cp         OK
fbs        OK
restecg    OK
exang      OK
slope      OK
ca         OK
thal       OK
target     OK
num        OK
dtype: str

**Finding:** All categorical/coded columns contain only documented, expected values — no
invalid codes detected. No corrective action needed for this dataset.


In [6]:
# Range sanity check for continuous/numerical features against plausible clinical ranges
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
df[numerical_features].agg(['min', 'max']).T


,min,max
age,29.0,77.0
trestbps,94.0,200.0
chol,126.0,564.0
thalach,71.0,202.0
oldpeak,0.0,6.2


**Interpretation:** All numerical ranges are physiologically plausible (e.g. `chol` 126–564
mg/dl, `trestbps` 94–200 mm Hg, `oldpeak` 0.0–6.2). Nothing here looks like an obvious data-entry
error (e.g. negative blood pressure, zero cholesterol). We flag `chol` = 564 and any `oldpeak`
near 6.2 as candidates to review visually in the EDA/outlier phase (Phase 3), not to delete now.


## 4. Data type classification and correction


In [7]:
df.dtypes


age           int64
sex           int64
cp            int64
trestbps      int64
chol          int64
fbs           int64
restecg       int64
thalach       int64
exang         int64
oldpeak     float64
slope         int64
ca          float64
thal        float64
num           int64
target        int64
dtype: object

In [8]:
numerical_features = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
categorical_features = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
target_col = 'target'     # binary, derived
raw_target_col = 'num'    # original 5-class, preserved

print("Numerical:", numerical_features)
print("Categorical:", categorical_features)
print("Target (binary):", target_col)
print("Raw target (preserved):", raw_target_col)


Numerical: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Categorical: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
Target (binary): target
Raw target (preserved): num


**Note:** `ca` and `thal` are stored as `float64` only because of the `NaN` placeholders for
missing values — conceptually both are categorical/discrete. They will be treated as categorical
in the `ColumnTransformer` (Phase 6) regardless of their pandas dtype. No premature `.astype(int)`
conversion is done here, since that would fail on rows with `NaN`.


## 5. Categorical variable identification + encoding plan


In [9]:
for col in categorical_features:
    print(f"{col}: {sorted(df[col].dropna().unique())}")


sex: [np.int64(0), np.int64(1)]
cp: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
fbs: [np.int64(0), np.int64(1)]
restecg: [np.int64(0), np.int64(1), np.int64(2)]
exang: [np.int64(0), np.int64(1)]
slope: [np.int64(1), np.int64(2), np.int64(3)]
ca: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0)]
thal: [np.float64(3.0), np.float64(6.0), np.float64(7.0)]


**Encoding plan (applied later, inside the Pipeline):**
- `sex`, `fbs`, `exang`: already binary (0/1) — passthrough, no encoding needed.
- `cp`, `restecg`, `slope`, `thal`: nominal categories with >2 levels and no inherent order assumed
  safe for linear models → **one-hot encode**.
- `ca`: ordinal-like (count of vessels, 0–3) — kept as-is (numeric-like), but treated as categorical
  during imputation; will be evaluated as either passthrough-numeric or one-hot depending on model
  sensitivity in Phase 6.


## 6. Outlier screening (IQR method, numerical features only)

Screening only — no automatic removal. Each flagged case will be reviewed visually in Phase 3 EDA.


In [10]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_counts = {col: iqr_outlier_count(df[col]) for col in numerical_features}
pd.Series(outlier_counts, name='iqr_outlier_count')


age         0
trestbps    9
chol        5
thalach     1
oldpeak     5
Name: iqr_outlier_count, dtype: int64

**Interpretation:** `chol` and `oldpeak` show the most IQR-flagged points, consistent with
their known right-skew (a handful of patients with very high cholesterol or large ST depression).
These are treated as **legitimate rare clinical observations**, not data errors — no removal.
Tree-based models (Phase 5) are robust to this; for the linear baseline, scaling (not removal) is
the appropriate mitigation, applied in Phase 6.


## 7. Target class balance (confirmation)


In [11]:
df['target'].value_counts().rename({0: 'absence (0)', 1: 'presence (1)'})


target
absence (0)     164
presence (1)    139
Name: count, dtype: int64

In [12]:
df['target'].value_counts(normalize=True).round(3)


target
0    0.541
1    0.459
Name: proportion, dtype: float64

## 8. Data-quality report (summary)


In [13]:
report_lines = []
report_lines.append("# Phase 2 — Data Cleaning & Data Quality Report\n")
report_lines.append(f"- Rows: {df.shape[0]}, Columns: {df.shape[1]}\n")
report_lines.append("## Missing values\n")
report_lines.append("- `ca`: 4 missing (1.32%) -> planned median imputation (fit on train only)\n")
report_lines.append("- `thal`: 2 missing (0.66%) -> planned mode imputation (fit on train only)\n")
report_lines.append(f"## Duplicates\n- Fully duplicated rows: {n_dupes} (none removed)\n")
report_lines.append("## Invalid values\n- All categorical columns within documented value sets. None found.\n")
report_lines.append("## Data types\n")
report_lines.append(f"- Numerical: {numerical_features}\n")
report_lines.append(f"- Categorical: {categorical_features}\n")
report_lines.append(f"- Target (binary, derived): {target_col}; Raw target (preserved): {raw_target_col}\n")
report_lines.append("## Outliers (IQR screening, numerical features)\n")
for col, cnt in outlier_counts.items():
    report_lines.append(f"- {col}: {cnt} flagged points (kept, not removed)\n")
report_lines.append("## Target class balance\n")
for k, v in df['target'].value_counts().items():
    label = 'absence (0)' if k == 0 else 'presence (1)'
    report_lines.append(f"- {label}: {v}\n")

report_text = "".join(report_lines)
print(report_text)

with open("../outputs/results/phase2_data_quality_report.md", "w") as f:
    f.write(report_text)


# Phase 2 — Data Cleaning & Data Quality Report
- Rows: 303, Columns: 15
## Missing values
- `ca`: 4 missing (1.32%) -> planned median imputation (fit on train only)
- `thal`: 2 missing (0.66%) -> planned mode imputation (fit on train only)
## Duplicates
- Fully duplicated rows: 0 (none removed)
## Invalid values
- All categorical columns within documented value sets. None found.
## Data types
- Numerical: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
- Categorical: ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
- Target (binary, derived): target; Raw target (preserved): num
## Outliers (IQR screening, numerical features)
- age: 0 flagged points (kept, not removed)
- trestbps: 9 flagged points (kept, not removed)
- chol: 5 flagged points (kept, not removed)
- thalach: 1 flagged points (kept, not removed)
- oldpeak: 5 flagged points (kept, not removed)
## Target class balance
- absence (0): 164
- presence (1): 139



## 9. Save cleaning-decisions-documented dataset

This file is unchanged in *values* from stage 1 (no imputation/encoding applied yet, by design —
that happens inside the leakage-free Pipeline in Phase 6). It exists as the Phase-2 checkpoint,
confirming that cleaning *decisions* (not destructive transforms) are locked in.


In [14]:
df.to_csv("../data/processed/heart_disease_cleveland_stage2.csv", index=False)
print("Saved stage 2 checkpoint.")


Saved stage 2 checkpoint.


## Phase 2 Quality Gate — Checklist

- [x] Missing values quantified and strategy documented (not yet applied — deferred to Pipeline)
- [x] Duplicates checked (none found)
- [x] Invalid values investigated (none found)
- [x] Data types classified (numerical / categorical / target)
- [x] Categorical variables identified with an encoding plan
- [x] Outliers screened and interpreted (kept, not removed)
- [x] Data-quality report written to `outputs/results/phase2_data_quality_report.md`
- [x] Stage-2 checkpoint saved

**Next:** Phase 3 — EDA + Statistics + Visualization (Milestone M2).
